# Database Clear Utility
Safely clears one or both pipeline databases.
Each section is independent — run only the cells you need.

**Databases managed:**
- `trials.db` — trials, corpus, compounds, cleaning_log
- `pubmed_corpus.db` — PubMed abstracts (legacy, from trial_getter v1)

> ⚠️  All operations are irreversible. There is no undo.

In [49]:
# --- Config ---
import sqlite3
import os
import pandas as pd
from datetime import datetime

SQLITE_DB  = "trials_v3.db"
PUBMED_DB  = "pubmed_corpus.db"

print("✅ Config loaded")
print(f"   trials_v3.db   exists: {os.path.exists(SQLITE_DB)}")
print(f"   pubmed_corpus.db exists: {os.path.exists(PUBMED_DB)}")

✅ Config loaded
   trials_v3.db   exists: True
   pubmed_corpus.db exists: True


In [50]:
# --- Inspect Current State (run before clearing) ---

def inspect_db(db_path: str):
    if not os.path.exists(db_path):
        print(f"  {db_path} — does not exist")
        return
    size_kb = os.path.getsize(db_path) / 1024
    con = sqlite3.connect(db_path)
    tables = con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    print(f"\n📁 {db_path}  ({size_kb:.1f} KB)")
    for (tbl,) in tables:
        count = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
        print(f"   {tbl:<30} {count:>6} rows")
    con.close()

inspect_db(SQLITE_DB)
inspect_db(PUBMED_DB)


📁 trials_v3.db  (176.0 KB)
   cleaning_log                       12 rows
   compound_abstracts                  2 rows
   compounds                          27 rows
   corpus                             20 rows
   corpus_staging                     20 rows
   sqlite_sequence                     1 rows
   trial_compounds                    23 rows
   trials                             20 rows
   trials_staging                     20 rows

📁 pubmed_corpus.db  (53884.0 KB)
   abstracts                       17951 rows


In [51]:
# --- Clear Individual Tables in trials_vX.db ---
# Uncomment the tables you want to clear.
# Clearing compounds also clears cleaning_log for consistency.

con = sqlite3.connect(SQLITE_DB)

tables_to_clear = [
    # "trials",        # raw AACT data — re-run GUI to repopulate
    # "corpus",        # formatted trial corpus — re-run build_corpus()
    # "compounds",     # compounds registry + all cleaning state
    # "cleaning_log",  # efficacy monitor log
]

if not tables_to_clear:
    print("⚠️  No tables selected — uncomment the tables you want to clear")
else:
    for tbl in tables_to_clear:
        try:
            con.execute(f"DELETE FROM {tbl}")
            print(f"✅ Cleared: {tbl}")
        except Exception as e:
            print(f"⚠️  Could not clear {tbl}: {e}")
    con.commit()
    print("\n✅ Selected tables cleared")
    print("   Re-run the relevant cells in trial_getter_2.ipynb to repopulate")

con.close()

⚠️  No tables selected — uncomment the tables you want to clear


In [52]:
# --- Reset Cleaning State Only ---
# Clears canonical_name, exclude, confidence, cleaned_by, pubmed_context
# from the compounds table — preserving compound_name and search_term.
# Use this to re-run the full cleaning chain without re-fetching from AACT.
# Also clears cleaning_log for the selected search term.

# Set the search term to reset (must match exactly what was used in the GUI)
SEARCH_TERM = ""   # ← fill in, e.g. "myeloma"

if not SEARCH_TERM.strip():
    print("⚠️  Set SEARCH_TERM before running")
else:
    term = SEARCH_TERM.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    con.execute("""
        UPDATE compounds
        SET canonical_name = NULL,
            exclude        = 0,
            exclude_reason = NULL,
            confidence     = NULL,
            cleaned_by     = NULL,
            pubmed_context = NULL
        WHERE search_term = ?
    """, (term,))

    con.execute("""
        DELETE FROM cleaning_log WHERE search_term = ?
    """, (term,))

    con.commit()

    count = con.execute(
        "SELECT COUNT(*) FROM compounds WHERE search_term = ?", (term,)
    ).fetchone()[0]
    con.close()

    print(f"✅ Cleaning state reset for '{term}'")
    print(f"   {count} compounds ready for re-run")
    print(f"   Trials and corpus preserved — no need to re-fetch from AACT")

⚠️  Set SEARCH_TERM before running


In [53]:
# --- Delete trials.db Entirely ---
# Removes the entire database file. Run init_sqlite() in trial_getter_2
# to recreate it from scratch.
# Uncomment to enable.

if os.path.exists(SQLITE_DB):
    os.remove(SQLITE_DB)
    print(f"✅ {SQLITE_DB} deleted")
    print("   Run init_sqlite() in trial_getter_3.ipynb to recreate")
else:
    print(f"⚠️  {SQLITE_DB} not found")

✅ trials_v3.db deleted
   Run init_sqlite() in trial_getter_3.ipynb to recreate


In [54]:
# --- Clear pubmed_corpus.db (legacy v1 database) ---
# This database was used by trial_getter v1 for disease-level PubMed corpus.
# In v2, PubMed context is stored per-compound in compounds.pubmed_context.
# Safe to delete if you no longer need the v1 corpus.
# Uncomment to enable.

if os.path.exists(PUBMED_DB):
    os.remove(PUBMED_DB)
    print(f"✅ {PUBMED_DB} deleted")
else:
    print(f"⚠️  {PUBMED_DB} not found")

✅ pubmed_corpus.db deleted


In [55]:
# --- Clear All Data for a Specific Search Term ---
# Removes all trials, corpus entries, compounds, and cleaning log
# for one disease search term. Useful when re-running a disease from scratch
# without disturbing data for other diseases in the same database.

# Set the search term to remove
SEARCH_TERM = ""   # ← fill in, e.g. "myeloma"

if not SEARCH_TERM.strip():
    print("⚠️  Set SEARCH_TERM before running")
else:
    term = SEARCH_TERM.lower().strip()
    con  = sqlite3.connect(SQLITE_DB)

    # Count before
    before = {
        "trials"      : con.execute("SELECT COUNT(*) FROM trials WHERE search_term=?",      (term,)).fetchone()[0],
        "compounds"   : con.execute("SELECT COUNT(*) FROM compounds WHERE search_term=?",   (term,)).fetchone()[0],
        "cleaning_log": con.execute("SELECT COUNT(*) FROM cleaning_log WHERE search_term=?",(term,)).fetchone()[0],
    }
    corpus_ncts = [r[0] for r in con.execute(
        "SELECT nct_id FROM trials WHERE search_term=?", (term,)
    ).fetchall()]

    # Delete
    con.execute("DELETE FROM trials WHERE search_term=?",       (term,))
    con.execute("DELETE FROM compounds WHERE search_term=?",     (term,))
    con.execute("DELETE FROM cleaning_log WHERE search_term=?",  (term,))
    if corpus_ncts:
        placeholders = ",".join(["?"]*len(corpus_ncts))
        con.execute(f"DELETE FROM corpus WHERE nct_id IN ({placeholders})",
                    corpus_ncts)
    con.commit()
    con.close()

    print(f"✅ All data cleared for search term: '{term}'")
    for tbl, cnt in before.items():
        print(f"   {tbl:<20} {cnt} rows removed")
    print(f"   corpus                {len(corpus_ncts)} rows removed")

⚠️  Set SEARCH_TERM before running


In [56]:
# --- Inspect After Clearing ---
inspect_db(SQLITE_DB)
inspect_db(PUBMED_DB)

  trials_v3.db — does not exist
  pubmed_corpus.db — does not exist
